In [1]:
import pandas as pd
import re
import logging
import os
# Configure logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

DEFAULT_NAMESPACE = "uri://ed-fi.org"

# Load descriptors CSV once and build a lookup table.
# The CSV file is assumed to have these columns:
# DESCRIPTOR_NAME,Owner,NAMESPACE,CODE_VALUE,SHORT_DESCRIPTION,DESCRIPTION
# Load descriptors CSV once and build a lookup table.
def load_descriptor_lookup(csv_path):
    """
    Load descriptors from CSV and create a case-insensitive lookup table.
    The CSV file is assumed to have these columns:
    DESCRIPTOR_NAME,Owner,NAMESPACE,CODE_VALUE,SHORT_DESCRIPTION,DESCRIPTION
    """
    try:
        encodings_to_try = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']
        df = None
        
        for encoding in encodings_to_try:
            try:
                df = pd.read_csv(csv_path, encoding=encoding, dtype=str, keep_default_na=False)
                #print(f"Successfully read descriptors CSV with encoding: {encoding}")
                break
            except UnicodeDecodeError:
                if encoding == encodings_to_try[-1]:
                    raise Exception(f"Could not read descriptors CSV with any encoding")
                continue
                
        if df is None:
            raise Exception("Failed to load descriptors CSV")
            
        lookup = {}
        descriptor_mapping = {
            'race_descriptors': ['race_descriptors', 'aggregated_race_descriptors'],
            'sex_descriptors': ['sex_descriptors', 'sex_type_descriptors'],
            'credit_type_descriptors': ['credit_type_descriptors', 'available_credit_type_descriptors'],
            'course_gpa_applicability_descriptors': ['course_g_p_a_applicability_descriptors', 'gpa_applicability_descriptors'],
            
        }
        
        # Build a key: (descriptor_name, code_value_lower) -> "NAMESPACE#CODE_VALUE"
        for _, row in df.iterrows():
            descriptor_name = row['DESCRIPTOR_NAME']
            code_value = row['CODE_VALUE']
            namespace = row['NAMESPACE']
            
            # Create descriptor value with namespace
            descriptor_value = f"{namespace}#{code_value}"
            
            # Add the primary mapping (case insensitive)
            lookup[(descriptor_name, code_value)] = descriptor_value
            
            # Add alternative descriptor names if mapped
            for primary, alternatives in descriptor_mapping.items():
                if descriptor_name in alternatives:
                    for alt_name in alternatives:
                        if alt_name != descriptor_name:  # Skip if it's the same
                            lookup[(alt_name, code_value)] = descriptor_value
        
        #print(f"Loaded {len(df)} descriptors into lookup table")
        return lookup
        
    except Exception as e:
        print(f"Error loading descriptor lookup: {e}")
        return {}

# Cache the lookup table from the Descriptors.csv file.
DESCRIPTOR_LOOKUP = load_descriptor_lookup('./data/Descriptors.csv')



def normalize_descriptor_field(field_path):
    """
    Normalize a descriptor field path to match the format in the descriptors CSV.
    Only extracts the final descriptor token name from complex paths.
    
    Examples:
    - /ed-fi/staffs/sexDescriptor -> sex_descriptors
    - /ed-fi/staffs/races[n].raceDescriptor -> race_descriptors
    - categories[0].educationOrganizationCategoryDescriptor -> education_organization_category_descriptors
    """
    # Step 1: Extract the last descriptor segment 
    # First split by '/' and take the last part
    last_segment = field_path.split('/')[-1]
    
    # Then handle array notation by splitting on '].' if present
    if '[' in last_segment and '].' in last_segment:
        last_segment = last_segment.split('].')[-1]
    elif '.' in last_segment:
        # Handle dot notation without array brackets
        last_segment = last_segment.split('.')[-1]
    
    # Step 2: Convert from camelCase to snake_case
    normalized = re.sub(r'(?<!^)(?=[A-Z])', '_', last_segment).lower()
    
    # Exception: Handle "GPA" specifically to avoid breaking it into "g_p_a"
    normalized = normalized.replace('_g_p_a_', '_gpa_')
    
    # Special case: Ignore prefixes 'maximum' and 'minimum' for credit type descriptors
    if normalized.startswith('maximum_') or normalized.startswith('minimum_'):
        normalized = normalized.replace('maximum_', '').replace('minimum_', '')
    
    # Step 3: Ensure it ends with 's' for plural form
    if not normalized.endswith('s') and normalized.endswith('descriptor'):
        normalized = normalized[:-10] + 'descriptors'  # Replace "descriptor" with "_descriptors"
    elif not normalized.endswith('s'):
        normalized += 's'
    
    return normalized

def update_row_descriptors(row):
    """
    For each column that looks like it contains a descriptor, normalize the field name
    to match the format in the descriptors CSV.
    """
    no_match = False
    for col in row.index:
        if "Descriptor" in col:
            # Only process non-empty values
            if pd.notna(row[col]) and str(row[col]).strip():
                # Normalize the descriptor field name
                normalized_field = normalize_descriptor_field(col)
                #if the value is numeric, convert it to str(int)
                if isinstance(row[col], (int, float)):
                    value = str(int(row[col]))
                else:
                    value = str(row[col]).strip()
                
                lookup_key = (normalized_field, value)
                
                # Special case for discipline descriptor
                if normalized_field == 'discipline_descriptors':
                    # Use the Boston Public Schools namespace for this descriptor
                    row[col] = "uri://mybps.org/DisciplineDescriptor#{value}".format(value=value)
                    continue
                
                # Debug output to verify correct normalization
               # print(f"Field: {col} → Normalized: {normalized_field}, Value: {value}")
                
                if lookup_key in DESCRIPTOR_LOOKUP:
                    row[col] = DESCRIPTOR_LOOKUP[lookup_key]
                else:
                    no_match = True
                    logger.warning(f"No match for descriptor: '{col}' → '{normalized_field}' with value '{row[col]}' (key: {lookup_key})")
            # Empty values remain unchanged
    return row, no_match

def process_csv(input_csv, output_dir, no_match_csv, output_csv):
    df = pd.read_csv(input_csv,dtype=str, keep_default_na=False)
    updated_rows = []
    no_match_rows = []
    
    for index, row in df.iterrows():
        row_updated, flag = update_row_descriptors(row.copy())
        updated_rows.append(row_updated)
        if flag:
            no_match_rows.append(row_updated)
            #if the number of rows count is greater than 1000 terminate the loop
        if len(no_match_rows) > 1000:
            logger.warning("Too many rows with no descriptor match, stopping processing to avoid excessive logging.")
            logger.warning(f"Total no match rows: {len(no_match_rows)}")
            #write out the no match rows to csv
            no_match_df = pd.DataFrame(no_match_rows)
            no_match_df.to_csv(no_match_csv, index=False)
            break
    
    updated_df = pd.DataFrame(updated_rows)
    
      # Split the updated DataFrame by SchoolYear and save each to a separate CSV file
    for school_year, group in updated_df.groupby('SchoolYear'):
        file_dir = os.path.join(output_dir, str(school_year))
        if not os.path.exists(file_dir) or not os.path.isdir(file_dir):
         os.makedirs(file_dir, exist_ok=True)
        output_csv_path = os.path.join(file_dir, f"{output_csv}")
        group.to_csv(output_csv_path, index=False)
        logger.info(f"Updated rows saved to {output_csv_path}")
    
    if no_match_rows:
        no_match_df = pd.DataFrame(no_match_rows)
        no_match_df.to_csv(no_match_csv, index=False)
        logger.info(f"Rows with no descriptor match saved to {no_match_csv}")
    else:
        logger.info("All rows had matching descriptor entries.")



In [2]:
def process_all_folders(data_dir, output_base_dir):
    for root, dirs, files in os.walk(data_dir):
        for dir_name in sorted(dirs):
            #Print for debugging
            print(f"Processing directory: {dir_name}")
            print(f"Output base dir: {output_base_dir}")
            input_csv = os.path.join(data_dir, dir_name, f"{dir_name}.csv")
            output_dir = os.path.join(output_base_dir, dir_name)
            no_match_csv = os.path.join(output_dir, f"NoMatch{dir_name}.csv")
            output_csv = f"{dir_name}.csv"
            print(f"Final output dir: {output_dir}")
            
            if os.path.exists(input_csv):
                os.makedirs(output_dir, exist_ok=True)
                process_csv(input_csv, output_dir, no_match_csv,output_csv)
            else:
                logger.warning(f"Input CSV not found: {input_csv}")

In [3]:
data_dir = './data'
output_base_dir = './output'
process_all_folders(data_dir, output_base_dir)

Processing directory: calendarDates
Output base dir: ./output
Final output dir: ./output/calendarDates


INFO: Updated rows saved to ./output/calendarDates/2012/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2013/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2014/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2015/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2016/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2017/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2018/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2019/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2020/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2021/calendarDates.csv
INFO: Updated rows saved to ./output/calendarDates/2022/calendarDates.csv
INFO: All rows had matching descriptor entries.


Processing directory: calendars
Output base dir: ./output
Final output dir: ./output/calendars


INFO: Updated rows saved to ./output/calendars/2007/calendars.csv
INFO: Updated rows saved to ./output/calendars/2008/calendars.csv
INFO: Updated rows saved to ./output/calendars/2009/calendars.csv
INFO: Updated rows saved to ./output/calendars/2012/calendars.csv
INFO: Updated rows saved to ./output/calendars/2013/calendars.csv
INFO: Updated rows saved to ./output/calendars/2014/calendars.csv
INFO: Updated rows saved to ./output/calendars/2015/calendars.csv
INFO: Updated rows saved to ./output/calendars/2016/calendars.csv
INFO: Updated rows saved to ./output/calendars/2017/calendars.csv
INFO: Updated rows saved to ./output/calendars/2018/calendars.csv
INFO: Updated rows saved to ./output/calendars/2019/calendars.csv
INFO: Updated rows saved to ./output/calendars/2020/calendars.csv
INFO: Updated rows saved to ./output/calendars/2021/calendars.csv
INFO: Updated rows saved to ./output/calendars/2022/calendars.csv
INFO: All rows had matching descriptor entries.


Processing directory: classPeriods
Output base dir: ./output
Final output dir: ./output/classPeriods


...

INFO: Updated rows saved to ./output/sessions/2018/sessions.csv
INFO: Updated rows saved to ./output/sessions/2019/sessions.csv
INFO: Updated rows saved to ./output/sessions/2020/sessions.csv
INFO: Updated rows saved to ./output/sessions/2021/sessions.csv
INFO: Updated rows saved to ./output/sessions/2022/sessions.csv
INFO: All rows had matching descriptor entries.


Processing directory: staffs
Output base dir: ./output
Final output dir: ./output/staffs


INFO: Updated rows saved to ./output/staffs/2000/staffs.csv
INFO: Updated rows saved to ./output/staffs/2001/staffs.csv
INFO: Updated rows saved to ./output/staffs/2002/staffs.csv
INFO: Updated rows saved to ./output/staffs/2003/staffs.csv
INFO: Updated rows saved to ./output/staffs/2004/staffs.csv
INFO: Updated rows saved to ./output/staffs/2005/staffs.csv
INFO: Updated rows saved to ./output/staffs/2006/staffs.csv
INFO: Updated rows saved to ./output/staffs/2007/staffs.csv
INFO: Updated rows saved to ./output/staffs/2008/staffs.csv
INFO: Updated rows saved to ./output/staffs/2009/staffs.csv
INFO: Updated rows saved to ./output/staffs/2010/staffs.csv
INFO: Updated rows saved to ./output/staffs/2011/staffs.csv
INFO: Updated rows saved to ./output/staffs/2012/staffs.csv
INFO: Updated rows saved to ./output/staffs/2013/staffs.csv
INFO: Updated rows saved to ./output/staffs/2014/staffs.csv
INFO: Updated rows saved to ./output/staffs/2015/staffs.csv
INFO: Updated rows saved to ./output/sta

Processing directory: stateEducationAgencies
Output base dir: ./output
Final output dir: ./output/stateEducationAgencies
Processing directory: studentParentAssociations
Output base dir: ./output
Final output dir: ./output/studentParentAssociations


INFO: Updated rows saved to ./output/studentParentAssociations/1977/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1978/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1979/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1980/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1981/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1982/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1983/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1984/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1985/studentParentAssociations.csv
INFO: Updated rows saved to ./output/studentParentAssociations/1986/studentParentAssociations.csv
INFO: Updated rows s

In [4]:
import pandas as pd
import json
import re
import os 


def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        while len(obj[key]) <= index:
            obj[key].append({})
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)


def set_nested_value(obj, components, value):
    recursive_set(obj, components, value)


def map_row_to_json(row, numeric_columns=None, boolean_columns=None, double_columns=None):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    
    The "SchoolYear" column is ignored.
    """
    json_obj = {}
    numeric_columns = numeric_columns or []
    for col in row.index:
        if col == "SchoolYear":  # ignore the school year column
            continue
        if pd.notna(row[col]):
            path = re.sub(r'^/[^/]+/[^/]+[/.]', '', col)
            components = parse_path(path)
             # Convert numeric values to strings unless the column is in numeric_columns
            value = row[col]
            if col not in numeric_columns and isinstance(value, (int, float)) and not pd.isna(value):
                if isinstance(value, float) and value.is_integer():
                    value = str(int(value))
                else:
                    value = str(value)
            #if col name ends with any of the boolean columns and the value is a string, convert to boolean
            if any(col.endswith(boolean_col) for boolean_col in boolean_columns):
                if value.lower() == "true":
                    value = True
                elif value.lower() == "false":
                    value = False
                #also convert 1 to True and 0 to False
                elif value == "1":
                    value = True
                elif value == "0":
                    value = False
            if col.endswith("Name"):
                # Convert to string and strip whitespace
                value = str(value).strip()
            #if col name ends with any of the date columns and the value is a string, convert to date
            #if col name ends with any of the numeric columns and the value is a string, convert to int
            if any(col.endswith(numeric_col) for numeric_col in numeric_columns):
                #if the value is empty, set it to 0
                if value != "":
                    value = int(value)

            #if col name ends with any of the double columns and the value is a string, convert to double
            if any(col.endswith(double_col) for double_col in double_columns):
                value = float(value)    

            set_nested_value(json_obj, components, value)
    # Update descriptors in the resulting JSON object
    return json_obj


In [ ]:
from functools import partial


def convert_csv_to_jsonl(input_csv, output_jsonl):
    """
    Reads an updated CSV file from `input_csv`, applies the map_row_to_json function
    to each row to generate a JSON object, and writes each JSON object as a
    newline-delimited JSON (JSONL) file to `output_jsonl`.
    """
    import pandas as pd
    import json

    # Read the CSV file into a DataFrame.
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False, na_values=['','NULL'])
    # Convert each row to a JSON object using map_row_to_json (assumed to be defined).
    boolean_columns=['primaryEmailAddressIndicator','reportedToLawEnforcement','livesWith','highSchoolCourseRequirement']
    numeric_columns=['schoolId','schoolYear','stateEducationAgencyId','localEducationAgencyId','educationOrganizationNetworkId','sequenceOfCourse','educationServiceCenterId','contactPriority','educationOrganizationId','numberOfParts','periodSequence','maximumNumberOfSeats','totalInstructionalDays']
    double_columns=['availableCreditConversion','availableCredits','maximumAvailableCreditConversion','minimumAvailableCreditConversion','maximumAvailableCredits','minimumAvailableCredits','maximumCreditConversion','minimumCreditConversion','maximumCredits','minimumCredits','disciplineActionLength']
    map = partial(map_row_to_json,boolean_columns=boolean_columns, numeric_columns=numeric_columns, double_columns=double_columns)
    json_data = df.apply(map, axis=1).tolist()

    # Write the JSONL file.
    with open(output_jsonl, 'w') as f:
        for row_obj in json_data:
            f.write(json.dumps(row_obj) + "\n")

    print(f"JSONL data with updated descriptors (ignoring SchoolYear) saved to {output_jsonl}")

    
def convert_all_csv_to_jsonl(output_base_dir):
    for root, dirs, files in os.walk(output_base_dir):
        for file in files:
            if file.endswith('.csv'):
            
                input_csv = os.path.join(root, file)
                output_json = os.path.join(root, file.replace('.csv', '.jsonl'))
                convert_csv_to_jsonl(input_csv, output_json)

In [6]:

output_base_dir = './output'
convert_all_csv_to_jsonl(output_base_dir)

JSONL data with updated descriptors (ignoring SchoolYear) saved to ./output/educationOrganizationNetworkAssociations/2017/educationOrganizationNetworkAssociations.jsonl
JSONL data with updated descriptors (ignoring SchoolYear) saved to ./output/educationOrganizationNetworkAssociations/2015/educationOrganizationNetworkAssociations.jsonl
JSONL data with updated descriptors (ignoring SchoolYear) saved to ./output/educationOrganizationNetworkAssociations/2014/educationOrganizationNetworkAssociations.jsonl
JSONL data with updated descriptors (ignoring SchoolYear) saved to ./output/educationOrganizationNetworkAssociations/2022/educationOrganizationNetworkAssociations.jsonl
JSONL data with updated descriptors (ignoring SchoolYear) saved to ./output/educationOrganizationNetworkAssociations/2019/educationOrganizationNetworkAssociations.jsonl
JSONL data with updated descriptors (ignoring SchoolYear) saved to ./output/educationOrganizationNetworkAssociations/2018/educationOrganizationNetworkAssoc

In [ ]:

input_csv = './output/schoolYearTypes/UpdatedSchoolYearTypes.csv'
output_json = './output/schoolYearTypes/SchoolYearTypes.jsonl'
convert_csv_to_jsonl(input_csv, output_json)

In [ ]:
import json
import os
import glob

def process_calendar_file(input_file):
    """Process a single calendar JSONL file, converting schoolIds from string to int."""
    # Create output filename by inserting "Updated" before .jsonl extension
    base_dir = os.path.dirname(input_file)
    base_name = os.path.basename(input_file)
    if base_name.endswith('.jsonl'):
        file_name_without_ext = base_name[:-6]  # Remove .jsonl
        output_file = os.path.join(base_dir, f"Updated{base_name}")
    else:
        output_file = os.path.join(base_dir, f"Updated_{base_name}")
    
    print(f"Processing: {input_file}")
    print(f"Output to: {output_file}")
    
    records_processed = 0
    records_modified = 0
    
    with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
        for line_num, line in enumerate(fin, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                records_processed += 1
                
                # Flag to track if this record was modified
                modified = False
                
                # Convert schoolId from string to int if present
                if ("schoolReference" in record and 
                    "schoolId" in record["schoolReference"] and
                    isinstance(record["schoolReference"]["schoolId"], str)):
                    try:
                        record["schoolReference"]["schoolId"] = int(record["schoolReference"]["schoolId"])
                        modified = True
                        records_modified += 1
                    except ValueError:
                        print(f"  Warning: Unable to convert schoolId to integer in line {line_num}")
                
                fout.write(json.dumps(record) + "\n")
            except Exception as e:
                print(f"  Error processing line {line_num}: {e}")
    
    print(f"  Completed: {records_processed} records processed, {records_modified} records modified")
    return records_processed, records_modified


def main():
    # Base directory for calendar files
    base_dir = './output/calendars'
    
    # Find all year directories under the calendars directory
    year_dirs = [d for d in glob.glob(f"{base_dir}/*/") if os.path.isdir(d)]
    
    if not year_dirs:
        print(f"No year directories found in {base_dir}")
        # Check if there are JSONL files directly in the base directory
        jsonl_files = glob.glob(f"{base_dir}/*.jsonl")
        if jsonl_files:
            print(f"Found {len(jsonl_files)} JSONL files in base directory")
            for jsonl_file in jsonl_files:
                process_calendar_file(jsonl_file)
        else:
            print(f"No JSONL files found in {base_dir}")
        return
    
    # Process files in each year directory
    total_processed = 0
    total_modified = 0
    
    for year_dir in year_dirs:
        year = os.path.basename(os.path.dirname(year_dir))
        print(f"\nProcessing year directory: {year}")
        
        # Find all JSONL files in this year directory
        jsonl_files = glob.glob(f"{year_dir}/*.jsonl")
        
        if not jsonl_files:
            print(f"  No JSONL files found in {year_dir}")
            continue
        
        print(f"  Found {len(jsonl_files)} JSONL files")
        
        # Process each JSONL file
        for jsonl_file in jsonl_files:
            processed, modified = process_calendar_file(jsonl_file)
            total_processed += processed
            total_modified += modified
    
    print(f"\nTotal: {total_processed} records processed, {total_modified} records modified")

if __name__ == "__main__":
    main()

In [ ]:
with open('/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/all/data/classPeriods/class_periods.csv', 'r') as f_in, open('/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/all/data/classPeriods/class_periods_mod.csv', 'w') as f_out:
    # Track line number to handle header separately
    line_count = 0
    
    for line in f_in:
        line_count += 1
        
        # For header row (first line), write it unchanged
        if line_count == 1:
            f_out.write(line)
            continue
        
        # For data rows, quote all columns
        parts = line.strip().split(',')
        quoted_parts = [f'"{part}"' for part in parts]
        f_out.write(','.join(quoted_parts) + '\n')

In [ ]:
import pandas as pd
import os
import re

def unpivot_grade_levels(input_file, output_file):
    # Read the CSV file, skip comment lines
    with open(input_file, 'r') as f:
        first_line = f.readline().strip()
    
    skiprows = 1 if first_line.startswith('//') else 0
    
    # Read the data
    df = pd.read_csv(input_file, skiprows=skiprows, dtype=str)
    
    # Identify the grade level columns
    grade_level_cols = [col for col in df.columns if 'gradeLevels' in col]
    
    # Identify the key columns
    key_cols = [
        'SchoolYear', 
        '/ed-fi/schools/educationOrganizationCategories[0].educationOrganizationCategoryDescriptor',
        '/ed-fi/schools/schoolId',
        # Don't include the grade level columns here as they will be unpivoted
    ]
    
    # All other columns that aren't grade level columns or key columns
    other_cols = [col for col in df.columns if col not in grade_level_cols and col not in key_cols]
    
    # Unpivot the grade level columns
    melted_df = pd.melt(
        df,
        id_vars=key_cols + other_cols,
        value_vars=grade_level_cols,
        var_name='GradeLevelIndex',
        value_name='GradeLevel'
    )
    
    # Drop rows where GradeLevel is empty
    melted_df = melted_df.dropna(subset=['GradeLevel'])
    
    # Extract index number from GradeLevelIndex
    melted_df['GradeLevelIndex'] = melted_df['GradeLevelIndex'].apply(
        lambda x: re.search(r'\[(\d+)\]', x).group(1) if re.search(r'\[(\d+)\]', x) else '0'
    )
    
    # Sort by keys and grade level index
    melted_df = melted_df.sort_values(by=key_cols + ['GradeLevelIndex'])
    
    # Write to output file
    melted_df.to_csv(output_file, index=False)
    
    print(f"Processed {len(df)} schools with {len(melted_df)} grade level entries")
    print(f"Results saved to: {output_file}")
    
    return melted_df

# Get the directory of the input file
input_file = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/all/data/schools/schools.csv'
output_file = '/home/bruk/code/boston/earthmover_edfi_bundles/non_assessments/all/data/schools/schools_unpivoted_grades.csv'

# Run the function
unpivoted_data = unpivot_grade_levels(input_file, output_file)